In [1]:
import xarray as xr
from distributed import LocalCluster, Client

In [2]:
cluster = LocalCluster()
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:59488,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 16.00 GiB
Comm: tcp://127.0.0.1:59499,Total threads: 2
Dashboard: http://127.0.0.1:59502/status,Memory: 4.00 GiB
Nanny: tcp://127.0.0.1:59491,


In [3]:
ds_wb2 = xr.open_zarr(
    "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr",
    consolidated=True,
    storage_options=dict(token='anon', session_kwargs={'trust_env': True})
)

In [4]:
sliced_dataset = xr.concat((
    ds_wb2["geopotential"].sel(level=500, drop=True),
    ds_wb2["temperature"].sel(level=850, drop=True),
    ds_wb2["specific_humidity"].sel(level=700, drop=True),
    ds_wb2["u_component_of_wind"].sel(level=250, drop=True),
    ds_wb2["v_component_of_wind"].sel(level=250, drop=True),
    ds_wb2["2m_temperature"],
    ds_wb2["total_precipitation_6hr"]
), dim="var_name", coords="minimal")
sliced_dataset = sliced_dataset.transpose("time", "var_name", "longitude", "latitude")

In [ ]:
train_data = sliced_dataset.sel(time=slice("1979", "2017"))
val_data = sliced_dataset.sel(time=slice("2018", "2019"))
test_data = sliced_dataset.sel(time=slice("2020", "2022"))

In [ ]:
del sliced_dataset.encoding["chunks"]
del sliced_dataset.encoding["preferred_chunks"]

In [ ]:
train_data.to_zarr("data/wb2_train.zarr", mode="w")

/cerea_raid/users/finnt/usr/miniconda3/envs/patched_diffusion/lib/python3.10/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 10.56 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
val_data.to_zarr("data/wb2_val.zarr")
test_data.to_zarr("data/wb2_test.zarr")